In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [ ]:
# 1. Load dataset Adult Income
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
columns = ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status',
           'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss',
           'hours-per-week', 'native-country', 'income']
data = pd.read_csv(url, names=columns, skipinitialspace=True)

In [ ]:
# 2. Preprocessing: encode categorical variables dan hapus missing values
data = data.replace('?', np.nan).dropna()
le = LabelEncoder()
for col in data.select_dtypes(include=['object']).columns:
    data[col] = le.fit_transform(data[col])

X = data.drop('income', axis=1)
y = data['income']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [ ]:
# 3. Model overfit: Random Forest - parameter kompleks (tanpa regularisasi)
overfit_model = RandomForestClassifier(n_estimators=500, max_depth=None, min_samples_split=2, random_state=42)
overfit_model.fit(X_train, y_train)

RandomForestClassifier(n_estimators=500, random_state=42)

In [ ]:
# 4. Evaluasi akurasi pada data pelatihan dan data uji
train_pred = overfit_model.predict(X_train)
test_pred = overfit_model.predict(X_test)
print(f"Akurasi pada data pelatihan: {accuracy_score(y_train, train_pred):.4f}")
print(f"Akurasi pada data uji: {accuracy_score(y_test, test_pred):.4f}")

Akurasi pada data pelatihan: 1.0000
Akurasi pada data uji: 0.8572


In [ ]:
# 5. Simulasi bias amplification: analisis prediksi berdasarkan gender
X_test_with_sex = X_test.copy()
X_test_with_sex['income_pred'] = test_pred
X_test_with_sex['sex'] = X_test['sex']

# Hitung tingkat "penghasilan tinggi" yang diprediksi per gender
male_high_income = X_test_with_sex[(X_test_with_sex['sex'] == 1) & (X_test_with_sex['income_pred'] == 1)].shape[0]
female_high_income = X_test_with_sex[(X_test_with_sex['sex'] == 0) & (X_test_with_sex['income_pred'] == 1)].shape[0]
male_total = X_test_with_sex[X_test_with_sex['sex'] == 1].shape[0]
female_total = X_test_with_sex[X_test_with_sex['sex'] == 0].shape[0]

print(f"Persentase pria diprediksi berpenghasilan tinggi: {male_high_income / male_total * 100:.2f}%")
print(f"Persentase wanita diprediksi berpenghasilan tinggi: {female_high_income / female_total * 100:.2f}%")

Persentase pria diprediksi berpenghasilan tinggi: 27.55%
Persentase wanita diprediksi berpenghasilan tinggi: 9.10%
